# The Grounded Football Tactics & Code Explainer

This project uses Retrieval Augmented Generation (RAG) to fetch factual context about football tactics and coding algorithms before generating our explanatory analogy. This implements Project Idea 1 from the brainstorming session.

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import gradio as gr

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

## Knowledge Base
Here, I'm scraping some Wikipedia pages describing coding algorithms and football tactics to form our knowledge base.

In [ ]:
def scrape_wikipedia(title, filename):
    url = f"https://en.wikipedia.org/wiki/{title}"
    headers = {"User-Agent": "Mozilla/5.0 (educational project)"}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Remove noisy elements like mathematical formulas and citations
        for element in soup(["math", "sup", "style", "script"]):
            element.decompose()
            
        paragraphs = soup.find_all('p')
        text = '\n'.join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(text)
        print(f"Saved {title} to {filename}")
    else:
        print(f"Failed to retrieve {title} (status {response.status_code})")

os.makedirs('knowledge-base', exist_ok=True)
topics = [
    ("Tiki-taka", "tiki_taka.txt"),
    ("Gegenpressing", "gegenpressing.txt"),
    ("Catenaccio", "catenaccio.txt"),
    ("Total_Football", "total_football.txt"),
    ("Sliding_window_protocol", "sliding_window.txt"),
    ("Binary_search_algorithm", "binary_search.txt"),
    ("Depth-first_search", "dfs.txt"),
    ("Breadth-first_search", "bfs.txt")
]
for topic, filename in topics:
    path = os.path.join('knowledge-base', filename)
    if not os.path.exists(path):
        scrape_wikipedia(topic, path)

## Load and Chunk Documents
I'm using LangChain to load the text files and split them into chunks for vector search.

In [ ]:
loader = DirectoryLoader('knowledge-base/', glob="**/*.txt", loader_cls=TextLoader)
documents = loader.load()
print(f"Loaded {len(documents)} documents.")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks.")


## Vector Store

In [ ]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store created and stored locally.")


## RAG Generation Pipeline

In [ ]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.2)

def generate_grounded_analogy(pattern_name, tactical_style):
    query = f"{pattern_name} algorithm vs {tactical_style} football tactics"
    
    # Retrieve context
    retrieved_docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    system_prompt = f"""
You are an expert software engineering mentor and an astute football tactician.
Your goal is to explain the algorithmic coding pattern: '{pattern_name}'.
You must explain the concept using a detailed analogy based on the football tactic/style: '{tactical_style}'.

You MUST use the following factual context retrieved from our knowledge base to ensure your analogy is grounded in real historical and technical facts.
Context:
---
{context}
---

Structure your response as follows:
1. **The Core Concept**: Very briefly explain what the coding pattern is.
2. **The Pitch Analogy**: Introduce the football analogy using the factual context provided.
3. **How It Plays Out**: Map the elements of the coding pattern to the tactical movements on the pitch.
4. **The Playbook (Code)**: Provide a clean Python code skeleton of the pattern.
"""

    response = llm.invoke([{"role": "system", "content": system_prompt}])
    return response.content, context


## Gradio Interface

In [ ]:
def ui_wrapper(pattern, tactic):
    try:
        answer, context_used = generate_grounded_analogy(pattern, tactic)
        return answer, context_used
    except Exception as e:
        return f"Error: {str(e)}", ""

with gr.Blocks() as demo:
    gr.Markdown("# The Grounded Football & Code Explainer")
    gr.Markdown("Select a coding pattern and a football tactical style. Our RAG pipeline will fetch real Wikipedia context and generate a grounded, factual analogy!")
    
    with gr.Row():
        pattern_dropdown = gr.Dropdown(
            choices=["Sliding Window", "Binary Search", "Depth-First Search", "Breadth-First Search"],
            label="Coding Pattern"
        )
        tactic_dropdown = gr.Dropdown(
            choices=["Tiki-taka", "Gegenpressing", "Catenaccio", "Total Football"],
            label="Football Tactic"
        )
    
    btn = gr.Button("Generate Grounded Analogy")
    
    with gr.Row():
        output_analogy = gr.Markdown(label="Generated Analogy")
        output_context = gr.Textbox(label="Retrieved Context (from VectorDB)", lines=15)
        
    btn.click(fn=ui_wrapper, inputs=[pattern_dropdown, tactic_dropdown], outputs=[output_analogy, output_context])

demo.launch(inbrowser=True)


## Testing

In [ ]:
def test_retrieval_and_generation():
    print("Running tests...")
    
    # Test retrieval keyword matching
    docs = retriever.invoke("Tiki-taka")
    assert len(docs) > 0, "Should retrieve at least one document"
    
    content = " ".join([d.page_content.lower() for d in docs])
    assert "tiki-taka" in content or "barcelona" in content, "Retrieved docs should mention Tiki-taka or Barcelona"
    
    # Test generation
    answer, context = generate_grounded_analogy("Binary Search", "Tiki-taka")
    assert "Binary Search" in answer, "Answer should mention Binary Search"
    assert len(answer) > 200, "Answer should be a fully formed analogy"
    
    print("All tests passed!")

test_retrieval_and_generation()